In [2]:
import pandas as pd
import db_mgmt as mgmt
import os

In [3]:
os.getcwd()

'/Users/thiagorodrigues/Documents/CANOE/CANOE_ref'

In [4]:
data_schema = 'dbs/canoe_dataset_schema 4.sql'
db_file = 'dbs/canoe_transport.sqlite'
os.remove(db_file) if os.path.exists(db_file) else None
mgmt.convert_sql_to_sqlite(data_schema, db_file)
data = mgmt.sqlite_to_dfs(db_file)
data.keys()

dict_keys(['MetaData', 'MetaDataReal', 'SeasonLabel', 'SectorLabel', 'CapacityCredit', 'CapacityFactorProcess', 'CapacityFactorTech', 'CapacityToActivity', 'Commodity', 'CommodityType', 'ConstructionInput', 'CostEmission', 'CostFixed', 'CostInvest', 'CostVariable', 'Demand', 'DemandSpecificDistribution', 'EndOfLifeOutput', 'Efficiency', 'EfficiencyVariable', 'EmissionActivity', 'EmissionEmbodied', 'EmissionEndOfLife', 'ExistingCapacity', 'TechGroup', 'LoanLifetimeProcess', 'LoanRate', 'LifetimeProcess', 'LifetimeTech', 'Operator', 'LimitGrowthCapacity', 'LimitDegrowthCapacity', 'LimitGrowthNewCapacity', 'LimitDegrowthNewCapacity', 'LimitGrowthNewCapacityDelta', 'LimitDegrowthNewCapacityDelta', 'LimitStorageLevelFraction', 'LimitActivity', 'LimitActivityShare', 'LimitAnnualCapacityFactor', 'LimitCapacity', 'LimitCapacityShare', 'LimitNewCapacity', 'LimitNewCapacityShare', 'LimitResource', 'LimitSeasonalCapacityFactor', 'LimitTechInputSplit', 'LimitTechInputSplitAnnual', 'LimitTechOutput

In [5]:
dir = 'transport/inputs/'
transp = {}
for file in os.listdir(dir):
    #print(file)
    if file.endswith('.csv'):
        transp[file.split('_')[0]] = pd.read_csv(dir + file)


In [6]:
tech_to_remove = pd.read_csv('transport/Fuel_techs.csv')['tech'].to_list()
new_transp = {}
fuel = {}
transport = {}

for table, df in transp.items():
    if 'tech' in df.columns:
        fuel[table] = df[df['tech'].isin(tech_to_remove)].copy().reset_index(drop=True)
        new_transp[table] = df[~df['tech'].isin(tech_to_remove)].copy().reset_index(drop=True)
        print(table)
    else:
        new_transp[table] = df.copy()



ExistingCapacity
CapacityFactorTech
LimitTechInputSplitAnnual
LimitAnnualCapacityFactor
CostFixed
TechGroupMember
Technology
EmissionActivity
LifetimeTech
CostInvest
CapacityToActivity
LifetimeSurvivalCurve
CostVariable
Efficiency


In [7]:
comm = pd.read_csv('transport/Fuel_comm.csv')['commodity'].to_list()
comm



['T_h2_10', 'T_h2_100', 'T_elc_dc', 'T_ng', 'T_h2']

In [8]:
new_transp['Commodity'] = pd.read_csv('transport/Commodity.csv')
new_transp['CostFixed'] = pd.read_csv('dbs/CostFixed.csv')
new_transp['CostInvest'] = pd.read_csv('dbs/CostInvest.csv')
fuel['Commodity'] = pd.read_csv('transport/Commodity_fuel.csv')



In [9]:
for table, df in new_transp.items():
    for c in comm:
        if c in df.values:
            new_transp[table] = df[~df.isin([c]).any(axis=1)].copy().reset_index(drop=True)
            print(table)

print('\nfuel')
for table, df in fuel.items():
    for c in comm:
        if c in df.values:
            fuel[table] = df[df.isin([c]).any(axis=1)].copy().reset_index(drop=True)
            print(table)




fuel
LimitTechInputSplitAnnual
LimitTechInputSplitAnnual
LimitAnnualCapacityFactor
EmissionActivity
EmissionActivity
Efficiency
Efficiency
Efficiency
Efficiency
Efficiency
Commodity
Commodity
Commodity
Commodity


In [10]:
for table, df in new_transp.items():
    print(table)
    if ('data_id' in data[table].columns) and (table != 'DataSet'):
        if 'region' not in df.columns:
            df['data_id'] = 'TRPHR002'
        else:
            df['data_id'] = 'TRPHR' + df['region'].astype(str) + '002'

DataSource
ExistingCapacity
DataSet
CapacityFactorTech
LimitTechInputSplitAnnual
LimitAnnualCapacityFactor
CostFixed
TechGroupMember
Technology
EmissionActivity
LifetimeTech
CostInvest
Demand
CapacityToActivity
LifetimeSurvivalCurve
Commodity
CostVariable
TechGroup
Efficiency


In [11]:
new_transp['Technology']['tech'].to_list()

['T_BLND_DSL_ELC_HDV',
 'T_BLND_DSL_ELC_MDV',
 'T_BLND_ETH_GSL',
 'T_BLND_GSL_ELC_PHEV35',
 'T_BLND_GSL_ELC_PHEV50',
 'T_BLND_JTF',
 'T_BLND_RDSL_DSL',
 'T_BLND_SPK',
 'T_H2_HDV_REFUEL',
 'T_H2_LDV_REFUEL',
 'T_H2_MDV_REFUEL',
 'T_HDV_AJF_JFL',
 'T_HDV_AJF_SPK',
 'T_HDV_AJP_JFL',
 'T_HDV_AJP_SPK',
 'T_HDV_BIC_BEV',
 'T_HDV_BIC_DSL',
 'T_HDV_BIC_DSL_HEV',
 'T_HDV_BIC_FCEV',
 'T_HDV_BIC_GSL',
 'T_HDV_BS_BEV',
 'T_HDV_BS_CNG',
 'T_HDV_BS_DSL',
 'T_HDV_BS_DSL_HEV',
 'T_HDV_BS_DSL_PHEV',
 'T_HDV_BS_FCEV',
 'T_HDV_BS_FCHEV',
 'T_HDV_BS_GSL',
 'T_HDV_BT_BEV',
 'T_HDV_BT_CNG',
 'T_HDV_BT_DSL',
 'T_HDV_BT_DSL_HEV',
 'T_HDV_BT_DSL_PHEV',
 'T_HDV_BT_FCEV',
 'T_HDV_BT_FCHEV',
 'T_HDV_BT_GSL',
 'T_HDV_CHRG',
 'T_HDV_RF_DSL',
 'T_HDV_RF_H2',
 'T_HDV_RF_LNG',
 'T_HDV_RICP_DSL',
 'T_HDV_RICP_H2',
 'T_HDV_T_BEV',
 'T_HDV_T_DSL',
 'T_HDV_T_DSL_HEV',
 'T_HDV_T_DSL_PHEV',
 'T_HDV_T_FCEV',
 'T_HDV_T_FCHEV',
 'T_HDV_WTF_HFO',
 'T_HDV_WTF_LNG',
 'T_HDV_WTF_MDO',
 'T_LDV_BEV_CHRG',
 'T_LDV_C_BEV150',
 'T_LDV_

In [12]:
df = new_transp['LifetimeTech'].copy()


new = df.loc[df['tech'].str.endswith('_N'), 'tech'].to_list()
replacer = {tech: tech[:-2] for tech in new}

ex = df.loc[df['tech'].str.endswith('_EX'), 'tech'].to_list()
replacer.update({tech: tech[:-3] for tech in ex})
replacer

df = df.replace(replacer)

df.drop_duplicates(subset=['region', 'tech'])

new_transp['LifetimeTech'] = df.copy()

In [13]:
mgmt.update_sqlite(db_file, new_transp)


Inserting into DataSource with columns: source_id,source,notes,data_id
Inserting into ExistingCapacity with columns: region,tech,vintage,capacity,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into DataSet with columns: data_id,label,version,description,status,author,date,parent_id,changelog,notes
Inserting into CapacityFactorTech with columns: region,period,season,tod,tech,factor,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into LimitTechInputSplitAnnual with columns: region,period,input_comm,tech,operator,proportion,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into LimitAnnualCapacityFactor with columns: region,tech,vintage,output_comm,operator,factor,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into CostFixed with columns: region,period,tech,vintage,cost,data_id
Inserting into TechGroupMember with columns: group_name,tech,data_id
Inserting into Technolo

In [14]:
data = mgmt.sqlite_to_dfs('dbs/canoe_hr_4d.sqlite')


In [15]:
df_c = data['CostFixed']
df_e = data['Efficiency']


In [16]:
# Create a mask of rows in df_c that are NOT in df_e
# We use .merge or .isin on a combined string/tuple for multiple columns
cols = ['region', 'tech', 'vintage']

# This identifies which rows in df_c are NOT found in df_e
missing_in_e = df_c[~df_c[cols].apply(tuple, axis=1).isin(df_e[cols].apply(tuple, axis=1))]

print(missing_in_e[cols])


Empty DataFrame
Columns: [region, tech, vintage]
Index: []


In [17]:
rows_list = []
new_vintage = []

for r in missing_in_e['region'].unique():
    for t in missing_in_e['tech'].unique():
        sub_df = df_e.loc[(df_e['region'] == r) & (df_e['tech'] == t)]
        for v in missing_in_e.loc[(missing_in_e['region']==r) & (missing_in_e['tech']==t), 'vintage'].unique():
            if not sub_df.empty:
                # .iloc[[0]] returns a DataFrame with 1 row, preserving columns correctly
                rows_list.append(sub_df.iloc[[0]])
                new_vintage.append(v)
            

# Concat everything at once - much faster!
to_add = pd.concat(rows_list, ignore_index=True)
print(to_add)

ValueError: No objects to concatenate

In [19]:
to_add['vintage'] = new_vintage

NameError: name 'to_add' is not defined

In [ ]:
mgmt.update_sqlite('dbs/canoe_hr_4d.sqlite',{'Efficiency': to_add})

Inserting into Efficiency with columns: region,input_comm,tech,vintage,output_comm,efficiency,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id


In [20]:
data = mgmt.sqlite_to_dfs('dbs/canoe_hr_4d.sqlite')

In [21]:
df_sc = data['LifetimeSurvivalCurve']
df_lt = data['LifetimeTech']



In [22]:
# 1. Calculate the difference once
df_sc['period_diff'] = df_sc['period'] - df_sc['vintage']

# 2. Filter for fraction == 0 first to shrink the dataset
filtered_df = df_sc[df_sc['fraction'] == 0].copy()

# 3. Use groupby to find the minimum period_diff for each (region, tech)
# This keeps only the rows where the period_diff matches the group's minimum
new_SC = filtered_df[
    filtered_df.groupby(['region', 'tech'])['period_diff'].transform('min') == filtered_df['period_diff']
]

In [23]:
new_SC

# 2. Filter for fraction == 0 first to shrink the dataset
NEW_SC = pd.concat([df_sc.loc[df_sc['fraction'] != 0],new_SC], ignore_index=True)
NEW_SC = NEW_SC.drop(columns=['period_diff'])
new_SC


,region,period,tech,vintage,fraction,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id,period_diff
117120,AB,2042,T_LDV_C_BEV150,2010,0.0,From MOVES5,None,None,None,None,None,None,TRPHRAB002,32
117121,AB,2047,T_LDV_C_BEV150,2015,0.0,From MOVES5,None,None,None,None,None,None,TRPHRAB002,32
117122,AB,2052,T_LDV_C_BEV150,2020,0.0,From MOVES5,None,None,None,None,None,None,TRPHRAB002,32
117123,AB,2037,T_LDV_C_DSL,2005,0.0,From MOVES5,None,None,None,None,None,None,TRPHRAB002,32
117124,AB,2042,T_LDV_C_DSL,2010,0.0,From MOVES5,None,None,None,None,None,None,TRPHRAB002,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120075,SK,2067,T_MDV_T_GSL,2025,0.0,From MOVES5,None,None,None,None,None,None,TRPHRSK002,42
120076,SK,2072,T_MDV_T_GSL,2030,0.0,From MOVES5,None,None,None,None,None,None,TRPHRSK002,42
120077,SK,2077,T_MDV_T_GSL,2035,0.0,From MOVES5,None,None,None,None,None,None,TRPHRSK002,42
120078,SK,2082,T_MDV_T_GSL,2040,0.0,From MOVES5,None,None,None,None,None,None,TRPHRSK002,42


In [24]:
import sqlite3
import pandas as pd


conn = sqlite3.connect('dbs/canoe_hr_4d.sqlite')

# 1. Clear the table first
conn.execute("DELETE FROM LifetimeSurvivalCurve")

# 2. Add the new data
NEW_SC.to_sql('LifetimeSurvivalCurve', conn, if_exists='append', index=False)

conn.commit()
conn.close()

In [25]:
# 1. Ensure new_SC has unique pairs for the lookup
# We take the first occurrence of each region/tech
lookup = new_SC.drop_duplicates(subset=['region', 'tech']).set_index(['region', 'tech'])['period_diff']

# 2. Create the mask as before
mask = df_lt.set_index(['region', 'tech']).index.isin(lookup.index)

# 3. Update 'lifetime'
# Using .loc correctly to assign the mapped values back to the original dataframe
df_lt.loc[mask, 'lifetime'] = df_lt[mask].set_index(['region', 'tech']).index.map(lookup)

In [26]:
mgmt.update_sqlite('dbs/canoe_hr_4d.sqlite',{ 'LifetimeTech': df_lt})


Inserting into LifetimeTech with columns: region,tech,lifetime,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id


In [27]:
data = mgmt.sqlite_to_dfs('dbs/canoe_hr_4d.sqlite')



In [28]:
ex = data['ExistingCapacity']
ex

,region,tech,vintage,capacity,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,C_SPH_NG-EXS,2005,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
1,AB,C_SPH_NG-EXS,2010,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
2,AB,C_SPH_NG-EXS,2015,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
3,AB,C_SPH_NG-EXS,2020,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
4,AB,C_SPH_NG-EXS,2024,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2508,SK,T_MDV_T_GSL,2000,0.933600,k units,None,None,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
2509,SK,T_MDV_T_GSL,2005,10.593200,k units,None,None,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
2510,SK,T_MDV_T_GSL,2010,14.806000,k units,None,None,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
2511,SK,T_MDV_T_GSL,2015,12.347556,k units,None,None,NaN,NaN,NaN,NaN,NaN,TRPHRSK002


In [29]:
cf = data['CostFixed']
ex = data['ExistingCapacity']

# 1. Ensure new_SC has unique pairs for the lookup
# We take the first occurrence of each region/tech
lookup = ex.drop_duplicates(subset=['region', 'tech', 'vintage']).set_index(['region', 'tech', 'vintage'])

# 1. Identify rows NOT in lookup (you already did this)
mask_not_in_lookup = ~cf.set_index(['region', 'tech', 'vintage']).index.isin(lookup.index)

# 2. Identify rows where vintage is less than 2025
mask_old_vintage = cf['vintage'] < 2025

# 3. Combine them: Rows that are BOTH missing from lookup AND have vintage < 2025
# These are the rows you want to DISCARD
rows_to_drop_mask = mask_not_in_lookup & mask_old_vintage

# 4. Create NEW_CF by keeping everything ELSE
NEW_CF = cf[~rows_to_drop_mask].copy()
NEW_CF

,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,2025,C_SPH_NG-EXS,2005,0.177534,M$/PJ,Average maintenance cost of installed stock in...,C01,1.0,2.0,1.0,2.0,2.0,COMHRAB002
1,AB,2025,C_SPH_NG-EXS,2010,0.177534,M$/PJ,Average maintenance cost of installed stock in...,C01,1.0,2.0,1.0,2.0,2.0,COMHRAB002
2,AB,2030,C_SPH_NG-EXS,2010,0.177534,M$/PJ,Average maintenance cost of installed stock in...,C01,1.0,2.0,1.0,2.0,2.0,COMHRAB002
3,AB,2025,C_SPH_NG-EXS,2015,0.177534,M$/PJ,Average maintenance cost of installed stock in...,C01,1.0,2.0,1.0,2.0,2.0,COMHRAB002
4,AB,2030,C_SPH_NG-EXS,2015,0.177534,M$/PJ,Average maintenance cost of installed stock in...,C01,1.0,2.0,1.0,2.0,2.0,COMHRAB002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10463,SK,2040,T_MDV_CHRG,2035,1.643260,None,None,None,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
10464,SK,2045,T_MDV_CHRG,2035,1.643260,None,None,None,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
10465,SK,2040,T_MDV_CHRG,2040,1.643260,None,None,None,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
10466,SK,2045,T_MDV_CHRG,2040,1.643260,None,None,None,NaN,NaN,NaN,NaN,NaN,TRPHRSK002


In [30]:

conn = sqlite3.connect('dbs/canoe_hr_4d.sqlite')

# 1. Clear the table first
conn.execute("DELETE FROM CostFixed")

# 2. Add the new data
NEW_CF.to_sql('CostFixed', conn, if_exists='append', index=False)

conn.commit()
conn.close()

In [60]:
cv = data['CostVariable']
# ex = data['ExistingCapacity']

# 1. Ensure new_SC has unique pairs for the lookup
# We take the first occurrence of each region/tech
lookup = ex.drop_duplicates(subset=['region', 'tech', 'vintage']).set_index(['region', 'tech', 'vintage'])

# 1. Identify rows NOT in lookup (you already did this)
mask_not_in_lookup = ~cv.set_index(['region', 'tech', 'vintage']).index.isin(lookup.index)

# 2. Identify rows where vintage is less than 2025
mask_old_vintage = cv['vintage'] < 2025

# 3. Combine them: Rows that are BOTH missing from lookup AND have vintage < 2025
# These are the rows you want to DISCARD
rows_to_drop_mask = mask_not_in_lookup & mask_old_vintage

# 4. Create NEW_CF by keeping everything ELSE
NEW_CV = cv[~rows_to_drop_mask].copy()
NEW_CV


,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,2025,C_SPH_NG-EXS,2005,0.177534,M$/PJ,Average maintenance cost of installed stock in...,C01,1.0,2.0,1.0,2.0,2.0,COMHRAB002
1,AB,2025,C_SPH_NG-EXS,2010,0.177534,M$/PJ,Average maintenance cost of installed stock in...,C01,1.0,2.0,1.0,2.0,2.0,COMHRAB002
2,AB,2030,C_SPH_NG-EXS,2010,0.177534,M$/PJ,Average maintenance cost of installed stock in...,C01,1.0,2.0,1.0,2.0,2.0,COMHRAB002
3,AB,2025,C_SPH_NG-EXS,2015,0.177534,M$/PJ,Average maintenance cost of installed stock in...,C01,1.0,2.0,1.0,2.0,2.0,COMHRAB002
4,AB,2030,C_SPH_NG-EXS,2015,0.177534,M$/PJ,Average maintenance cost of installed stock in...,C01,1.0,2.0,1.0,2.0,2.0,COMHRAB002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10406,SK,2035,T_MDV_CHRG,2030,1.643260,None,None,None,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
10407,SK,2045,T_MDV_CHRG,2035,1.643260,None,None,None,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
10408,SK,2040,T_MDV_CHRG,2040,1.643260,None,None,None,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
10409,SK,2045,T_MDV_CHRG,2040,1.643260,None,None,None,NaN,NaN,NaN,NaN,NaN,TRPHRSK002


In [32]:

conn = sqlite3.connect('dbs/canoe_hr_4d.sqlite')

# 1. Clear the table first
conn.execute("DELETE FROM CostVariable")

# 2. Add the new data
NEW_CV.to_sql('CostVariable', conn, if_exists='append', index=False)

conn.commit()
conn.close()

In [33]:
data = mgmt.sqlite_to_dfs('dbs/canoe_hr_4d.sqlite')



In [34]:

for table, df in data.items():
    if 'vintage' in df.columns:
        data[table] = df.loc[df['vintage']< 2050]

for table, df, in data.items():
    if (table not in ['TimePeriod', 'LifetimeSurvivalCurve'])  and 'period' in df.columns:
        data[table] = data[table].loc[data[table]['period'] < 2050]

In [35]:
conn = sqlite3.connect('dbs/canoe_hr_4d.sqlite')

# 1. Clear the table first
for table, df in data.items():
    conn.execute(f"DELETE FROM {table}")

    # 2. Add the new data
    df.to_sql(f'{table}', conn, if_exists='append', index=False)

conn.commit()
conn.close()

In [36]:
data = mgmt.sqlite_to_dfs('dbs/canoe_hr_4d.sqlite')

In [ ]:
cap_f = data['LimitAnnualCapacityFactor'].copy()

lft = data['LifetimeTech'].copy()



In [38]:
cap_f.head()


,region,tech,vintage,output_comm,operator,factor,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,C_SPH_NG-EXS,2005,C_D_sph,le,0.254876,Mean hourly demand divided by peak hourly dema...,C02,1.0,2.0,5.0,2.0,3.0,COMHRAB002
1,AB,C_SPH_NG-EXS,2010,C_D_sph,le,0.254876,Mean hourly demand divided by peak hourly dema...,C02,1.0,2.0,5.0,2.0,3.0,COMHRAB002
2,AB,C_SPH_NG-EXS,2015,C_D_sph,le,0.254876,Mean hourly demand divided by peak hourly dema...,C02,1.0,2.0,5.0,2.0,3.0,COMHRAB002
3,AB,C_SPH_NG-EXS,2020,C_D_sph,le,0.254876,Mean hourly demand divided by peak hourly dema...,C02,1.0,2.0,5.0,2.0,3.0,COMHRAB002
4,AB,C_SPH_NG-EXS,2024,C_D_sph,le,0.254876,Mean hourly demand divided by peak hourly dema...,C02,1.0,2.0,5.0,2.0,3.0,COMHRAB002


In [39]:
lft.head()

,region,tech,lifetime,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,C_SPH_NG-EXS,23.0,Average life of installed stock indexed to sha...,C01,1.0,2.0,1.0,2.0,3.0,COMHRAB002
1,AB,C_SPH_ELC-EXS,12.0,Average life of installed stock indexed to sha...,C01,1.0,2.0,1.0,2.0,3.0,COMHRAB002
2,AB,C_SPHC_HP_AIR-NEW,21.0,Rounded life from AEO CDM ktekx technology men...,C01,1.0,2.0,1.0,2.0,3.0,COMHRAB002
3,AB,C_SPHC_HP_GEO-NEW,14.0,Rounded life from AEO CDM ktekx technology men...,C01,1.0,2.0,1.0,2.0,3.0,COMHRAB002
4,AB,C_SPH_ELC_BLR-NEW,15.0,Rounded life from AEO CDM ktekx technology men...,C01,1.0,2.0,1.0,2.0,3.0,COMHRAB002


In [40]:
df = data['TimePeriod'].copy()
start = df.loc[df['sequence']==0,'period'].iat[0]

In [41]:
ex = data['ExistingCapacity'].copy()

In [42]:
ex.head()

,region,tech,vintage,capacity,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,C_SPH_NG-EXS,2005,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
1,AB,C_SPH_NG-EXS,2010,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
2,AB,C_SPH_NG-EXS,2015,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
3,AB,C_SPH_NG-EXS,2020,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
4,AB,C_SPH_NG-EXS,2024,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002


In [54]:
lookup = lft.drop_duplicates(['region', 'tech']).set_index(['region', 'tech']).index
to_be_rm = []
for i in lookup:
    # Filter the data first
    vint_series = cap_f.loc[(cap_f['region'] == i[0]) & (cap_f['tech'] == i[1]), 'vintage']
    lt_series = lft.loc[(lft['region'] == i[0]) & (lft['tech'] == i[1]), 'lifetime']
    ex_ = ex.loc[(ex['region']== i[0]) & (ex['tech']==i[1])]

    # Only proceed if BOTH dataframes actually have the data
    if not vint_series.empty and not lt_series.empty:
        vint = vint_series.iat[0]
        lt_ = lt_series.iat[0]
        if vint < (start - lt_):
            to_be_rm.extend(cap_f.loc[(cap_f['region'] == i[0]) & (cap_f['tech'] == i[1]), 'vintage'].index.values)


            

In [55]:

data['LimitAnnualCapacityFactor'] = cap_f.loc[~cap_f.index.isin(to_be_rm)]


In [56]:
conn = sqlite3.connect('dbs/canoe_hr_4d.sqlite')

# 1. Clear the table first
for table, df in data.items():
    conn.execute(f"DELETE FROM {table}")

    # 2. Add the new data
    df.to_sql(f'{table}', conn, if_exists='append', index=False)

conn.commit()
conn.close()

In [57]:
data = mgmt.sqlite_to_dfs('dbs/canoe_hr_4d.sqlite')

ex = data['ExistingCapacity'].copy()
cf = data['LimitAnnualCapacityFactor'].copy()


In [58]:
ex.head()

,region,tech,vintage,capacity,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,C_SPH_NG-EXS,2005,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
1,AB,C_SPH_NG-EXS,2010,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
2,AB,C_SPH_NG-EXS,2015,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
3,AB,C_SPH_NG-EXS,2020,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002
4,AB,C_SPH_NG-EXS,2024,93.273698,(PJ/y),Secondary energy consumption shares calculated...,C05,1.0,2.0,2.0,2.0,1.0,COMHRAB002


In [59]:
lookfor = ex.drop_duplicates(['region', 'tech', 'vintage']).set_index(['region', 'tech', 'vintage']).index

In [61]:
df = pd.DataFrame({"A": [1, 2, 3], "B": [4, 5, 6], "C": [7, 8, 9]})

In [65]:
for idx, row in df.iterrows():
    print(row['A'], row['B'], row['C'])

1 4 7
2 5 8
3 6 9


In [66]:
for i in df.iterrows():
    print(i)

(0, A    1
B    4
C    7
Name: 0, dtype: int64)
(1, A    2
B    5
C    8
Name: 1, dtype: int64)
(2, A    3
B    6
C    9
Name: 2, dtype: int64)
